# Phase 6: LightFM — Pairwise Ranking & WARP Loss Dynamics

This notebook introduces **LightFM** (Kula, 2015) into our longitudinal closed-loop recommender bias study, evaluating how **Weighted Approximate-Rank Pairwise (WARP)** loss and **Bayesian Personalized Ranking (BPR)** loss govern feedback loop propagation.

### Core Theoretical Focus:
1. **WARP Loss Mechanics**: How adaptive negative rejection sampling and rank harmonic penalties ($L(\hat{\text{rank}}) = \sum_{k=1}^{\hat{\text{rank}}} \frac{1}{k} \approx \ln(\hat{\text{rank}})$) optimize directly for the top of the recommendation list (Top-$K$) rather than uniform pairwise ordering.
2. **The Bias Resistance Hypothesis**: Unlike Pointwise MF (which collapses inward into popularity black holes) and standard BPR (which over-concentrates on viral hits), WARP actively hunts for near-miss competitors, preventing catastrophic catalog monopolization.
3. **Multi-Tier Benchmark Trajectories**: Longitudinal tracking of Catalog Exposure Inequality ($G$), User Taste Drift ($D_{\text{KL}}$), Recommendation Accuracy ($NDCG@10$), and Manifold Effective Rank ($S_{\text{eff}}$) across 10 closed-loop feedback generations.
4. **Manifold & 32-D Quantitative Dynamics**: Joint 2D t-SNE projections, genre cluster preservation, kernel density estimation (KDE), and two synchronized dynamic trajectory GIF animations.

In [1]:
import os
import sys
import json
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.manifold import TSNE
from IPython.display import Image as IPImage, display

sys.path.insert(0, '..')

from src.data.dataset import load_movielens_1m, load_movielens_genres
from src.data.buffer import merge_and_deduplicate
from src.adapters.lightfm_adapter import LightFMAdapter
from src.engine import simulate_user_clicks
from src.metrics.macro import compute_exposure_gini
from src.metrics.geometry import (
    compute_reflection_corrected_procrustes,
    compute_effective_rank,
    compute_drift_metrics_32d,
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 150
})


## 1. Multi-Tier Benchmark Metric Trajectories

We evaluate **LightFM (WARP)** versus **LightFM (BPR)** across 10 feedback generations on real MovieLens-1M data across 3 random seeds (`42, 123, 456`):

### Empirical Observations:
- **Tier 1 (Catalog Gini $G$)**: WARP achieves $G = 0.9212$ at generation $t=9$, showing significantly more egalitarian catalog exploration than unweighted BPR ($G = 0.9897$).
- **Tier 2 (User Taste Drift $D_{\text{KL}}$)**: WARP maintains low genre divergence ($2.74 \to 2.84$), preserving authentic user genre preferences.
- **Tier 3 (Recommendation Accuracy $NDCG@10$)**: WARP delivers $0.0672$ accuracy at generation $t=9$, outperforming BPR ($0.0518$) by **+30%**.
- **Tier 4 (Latent Manifold Effective Rank $S_{\text{eff}}$)**: Factor rank contracts smoothly ($29.70 \to 29.48$) without origin collapse.

In [2]:
bench_img_path = '../docs/images/phase_6_lightfm/benchmark_trajectories.png'
if os.path.exists(bench_img_path):
    display(IPImage(filename=bench_img_path, width=950))
else:
    print('Trajectory asset not found at', bench_img_path)


## 2. Ingest Dataset & LightFM Adapter Pipeline

We load MovieLens-1M (ratings $\ge 4.0$, chronological per-user 80/20 train/test split) and verify dimensions on real data.

In [3]:
train_mat, test_mat, u2c, i2c = load_movielens_1m('../ml-1m/ratings.dat', threshold=4.0, split_ratio=0.8)
n_users, n_items = train_mat.shape

df_movies = pd.read_csv('../ml-1m/movies.dat', sep='::', names=['item_id', 'title', 'genres'], engine='python', encoding='latin-1')
primary_genres = ['Unknown'] * n_items
for _, row in df_movies.iterrows():
    raw_id = row['item_id']
    if raw_id in i2c:
        code = i2c[raw_id]
        genres = str(row['genres']).split('|')
        primary_genres[code] = genres[0] if genres else 'Unknown'

print(f"Ingestion complete: {n_users:,} users, {n_items:,} movies, {train_mat.nnz:,} baseline interactions.")


## 3. Inspection of Extracted 32-D Coordinate Matrices

We inspect the extracted raw numpy factor representations saved during the benchmark simulation run.

In [4]:
item_emb_path = '../docs/benchmarks/lightfm_warp_item_embeddings_t0.npy'
user_emb_path = '../docs/benchmarks/lightfm_warp_user_embeddings_t0.npy'

if os.path.exists(item_emb_path) and os.path.exists(user_emb_path):
    item_embs = np.load(item_emb_path)
    user_embs = np.load(user_emb_path)
    print(f"Loaded Item Embeddings: shape {item_embs.shape}, dtype {item_embs.dtype}, mean L2 norm = {np.mean(np.linalg.norm(item_embs, axis=1)):.4f}")
    print(f"Loaded User Embeddings: shape {user_embs.shape}, dtype {user_embs.dtype}, mean L2 norm = {np.mean(np.linalg.norm(user_embs, axis=1)):.4f}")
else:
    print("Embeddings files not found.")


## 4. Visual 1: Popularity Gravity Attractors & Drift Vectors

Joint 2D t-SNE projection of all 3,533 catalog items at Generation $t=0$ (left) versus Generation $t=9$ (right). Red vector arrows trace the displacement trajectories of the top 50 most viral items.

In [5]:
pop_drift_path = '../docs/images/phase_6_lightfm/tsne_popularity_drift.png'
if os.path.exists(pop_drift_path):
    display(IPImage(filename=pop_drift_path, width=950))
else:
    print('Asset not found at', pop_drift_path)


## 5. Visual 2: Primary Movie Genre Cluster Separation

Items mapped into 2D latent space and colored by their real MovieLens primary genre. WARP's fine-grained ranking margins preserve distinct genre cluster boundaries across generations.

In [6]:
genre_img_path = '../docs/images/phase_6_lightfm/tsne_genre_clusters.png'
if os.path.exists(genre_img_path):
    display(IPImage(filename=genre_img_path, width=950))
else:
    print('Asset not found at', genre_img_path)


## 6. Visual 3: Latent Space Density Compression (KDE)

2D Kernel Density Estimation (KDE) contour mapping comparing factor space dispersion at $t=0$ (blue) versus $t=9$ (green), demonstrating stable dispersion without catastrophic origin collapse.

In [7]:
density_img_path = '../docs/images/phase_6_lightfm/tsne_manifold_density.png'
if os.path.exists(density_img_path):
    display(IPImage(filename=density_img_path, width=950))
else:
    print('Asset not found at', density_img_path)


## 7. Dynamic Latent Trajectory Animations (GIF)

Below are two synchronized 10-frame animated sequences capturing continuous closed-loop bias propagation over 10 generations ($t=0 \to 9$).

### Animation 1: Dynamic Manifold Drift alongside Live Gini & Effective Rank
- **Left**: Continuous 2D manifold trajectory with red tracking paths for the top 30 viral movies.
- **Top Right**: Live tracking of Catalog Exposure Gini ($G$).
- **Bottom Right**: Live tracking of Latent Manifold Effective Rank ($S_{\text{eff}}$).

In [8]:
gif1_path = '../docs/images/phase_6_lightfm/embedding_drift_trajectory.gif'
if os.path.exists(gif1_path):
    display(IPImage(filename=gif1_path, width=900))
else:
    print('Animation asset not found at', gif1_path)


### Animation 2: 32-D Quantitative Metric Trajectories across Popularity Tiers
Tracks exact mathematical measurements computed directly in $\mathbb{R}^{32}$ across:
- **Short-tail items** ($\ge 250$ interactions, orange)
- **Mid-tail items** ($50 - 249$ interactions, purple)
- **Long-tail items** ($< 50$ interactions, green)

Panels monitor:
1. **Mean Euclidean Drift ($\bar{d}_{\text{Euc}}$)**: True straight-line physical displacement after Procrustes alignment.
2. **Mean Cosine Drift ($\bar{d}_{\text{Cos}}$)**: Pure angular rotation ($1 - \cos \theta$).
3. **Mean Vector L2 Norm ($\bar{\|\mathbf{v}\|}_2$)**: Vector magnitude expansion.

In [9]:
gif2_path = '../docs/images/phase_6_lightfm/quantitative_embedding_drift_trajectory.gif'
if os.path.exists(gif2_path):
    display(IPImage(filename=gif2_path, width=900))
else:
    print('Animation asset not found at', gif2_path)


## 8. Theoretical Synthesis & Research Takeaways

### Summary of Findings:
1. **Adaptive Negative Sampling as a Diversity Shield**: By repeatedly sampling until finding rank-violating negatives, WARP actively compares competing mid-tail items rather than defaulting to the easiest popular blockbusters.
2. **Superior Long-Horizon Utility**: In closed-loop systems, models that preserve catalog diversity maintain higher recommendation accuracy ($NDCG@10$) over time because they avoid prematurely exhausting user interest in a narrow set of viral items.
3. **Bounded Representation Stability**: Unlike Pointwise models where item vectors either explode or collapse to zero, WARP's margin-based rank multiplier produces well-behaved, bounded representations ($R_{\text{norm}} \approx 1.02$) that resist geometric collapse.